# Review the measured candidate regions

The breed-specific scans identified markers below the within-analysis cutoff with GEMMA’s Wald or likelihood-ratio tests. This chapter examines those candidates in their genomic and observed-genotype context. GEMMA remains the laboratory reporting workflow; GMMAT supplies a methodological comparison using the same dogs.

We examine neighboring measurements, marker correlation, genotype distributions, carrier sex and genetic similarity, and model-fit differences. These checks support manual review. They do not establish independent loci, causal variants, or reliable extreme-tail probabilities.

## Define the regions to examine

We include every Scheme A marker passing either GEMMA’s Wald or likelihood-ratio cutoff. Around each marker, we display 500,000 bases on either side. Overlapping windows on the same chromosome within a breed are combined so that the same neighborhood is not repeatedly plotted.

These are display regions, not declarations that signals are independent. Within each region, the selected candidate with the smallest Wald p-value is the display anchor. Choosing an anchor organizes the figure; it does not identify the causal variant. Scheme B results are retained for sensitivity comparison.

In [ ]:
from pathlib import Path
import sys
study_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/notebook_support.py").exists())
sys.path.insert(0, str(study_root / "scripts"))
from notebook_support import read_json, run_script, file_digest
review_folder = study_root / "results/private/version2/09_candidate_review"
run_script(study_root, "review_gwas_candidates.py", "--results-dir", review_folder)
review = read_json(review_folder / "summary.private.json")
assert review["status"] == "candidate_regions_reviewed"
assert review["original_sources_unchanged"]
assert review["review_config_sha256"] == file_digest(study_root / "config/candidate_review.json")
assert review["reporting_policy_sha256"] == file_digest(study_root / "config/gwas_reporting.json")
from IPython.display import Markdown, Image, display
lines = ["| Breed | Selected candidate markers | Display regions | Cases |",
         "| --- | ---: | ---: | ---: |"]
for row in review["flow"]:
    lines.append(f"| {row['breed']} | {row['selected_markers']} | {row['display_regions']} | {row['cases']} |")
display(Markdown("\n".join(lines)))

## Look for neighboring support

Regional plots show all tested measured markers inside each display region. Colors show how closely each marker’s measured genotypes track the display anchor, using squared correlation, or r². A value near one means the two markers carry very similar information in these dogs. A value near zero means little linear correlation. Missing measurements are omitted pairwise from this calculation.

Correlated neighboring markers can support a regional pattern, but they do not provide independent replications. An isolated point can reflect limited array coverage, a genuine narrow signal, or an unstable result; this review does not decide between those explanations.

In [ ]:
import numpy as np

def measured_r_squared(anchor, neighbor):
    """Compare two measured markers using only dogs observed at both."""
    observed = np.isfinite(anchor) & np.isfinite(neighbor)
    if observed.sum() < 3 or np.std(anchor[observed]) == 0 or np.std(neighbor[observed]) == 0:
        return None
    return float(np.corrcoef(anchor[observed], neighbor[observed])[0, 1] ** 2)

lines = ["| Region | Display anchor | Candidate markers | Measured neighbors | Other neighbors with r² ≥ 0.8 |",
         "| --- | --- | ---: | ---: | ---: |"]
for region in review["regions"]:
    lines.append(f"| {region['region']} | {region['anchor']} | {len(region['selected_markers'])} | "
                 f"{region['measured_neighbors']} | {region['nonanchor_neighbors_r2_ge_08']} |")
display(Markdown("\n".join(lines)))

## Examine the observed genotypes and carriers

Each regional figure also compares genotype proportions in cases and controls. The categories show actual observed genotypes; missing calls contribute to neither category nor denominator.

For descriptive carrier summaries, “carrier” means a dog with at least one copy of the less frequent allele at the anchor in Scheme A. This allele is not automatically the risk allele. The tables identify the counted carrier allele and split carriers by outcome and sex.

The genetic-similarity panel compares carrier pairs with all dog pairs, using the same chromosome-excluded relationship matrix supplied to the association models. Higher values indicate greater measured genetic similarity. Pair values share dogs and are not independent observations; this comparison is descriptive and does not reconstruct pedigrees or establish a distinct family.

In [ ]:
import csv
with (review_folder / "anchor_genotypes.private.tsv").open(encoding="utf-8", newline="") as handle:
    counts = list(csv.DictReader(handle, delimiter="\t"))
lines = ["| Region | Carrier allele | Case carriers, male / female | Control carriers, male / female |",
         "| --- | --- | ---: | ---: |"]
for region in review["regions"]:
    rows = [r for r in counts if r["region"] == region["region"]]
    def carrier_count(group, sex):
        return next(r["minor_allele_carriers"] for r in rows if r["group"] == group and r["sex"] == sex)
    lines.append(f"| {region['region']} | {region['carrier_allele']} | "
                 f"{carrier_count('case', 'male')} / {carrier_count('case', 'female')} | "
                 f"{carrier_count('control', 'male')} / {carrier_count('control', 'female')} |")
display(Markdown("\n".join(lines)))

In [ ]:
for breed in ["LR", "GSP"]:
    display(Markdown(f"## {breed}: candidate-region figures"))
    for region in review["regions"]:
        if region["breed"] != breed: continue
        display(Image(filename=str(review_folder / (region["region"] + ".png"))))

## Compare the fits and control definitions

The following table lists the anchor’s GEMMA effect and standard error, all three GEMMA p-values, and the GMMAT score p-value in both schemes. GEMMA’s effect is on the 0/1 outcome scale, not an odds ratio. A variance ratio at the optimizer’s search boundary is flagged for review.

Large differences between Wald, likelihood-ratio and score results are evidence that the statistical assessments differ, not independent confirmation of a region. The schemes also share the same cases. No model has been refitted or selected to make these candidates more significant.

In [ ]:
with (review_folder / "candidate_models.private.tsv").open(encoding="utf-8", newline="") as handle:
    models = list(csv.DictReader(handle, delimiter="\t"))
lines = ["| Region | Scheme | GEMMA β | SE | Wald p | Likelihood-ratio p | GEMMA score p | GMMAT score p | REML / ML boundary |",
         "| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |"]
for region in review["regions"]:
    for row in models:
        if row["region"] != region["region"] or row["marker"] != region["anchor"]: continue
        values = [f"{float(row[key]):.3g}" for key in ["GEMMA_beta", "GEMMA_SE", "GEMMA_wald_p", "GEMMA_lrt_p", "GEMMA_score_p", "GMMAT_score_p"]]
        boundary = row["GEMMA_remle_boundary"] + " / " + row["GEMMA_mle_boundary"]
        lines.append("| " + " | ".join([region["region"], row["scheme"], *values, boundary]) + " |")
display(Markdown("\n".join(lines)))

## How sensitive are the observed counts to one case?

For each anchor, we remove one case at a time from the descriptive allele-frequency calculation. The table gives the resulting range of case-minus-control ALT frequencies. This shows the leverage of an individual case on the observed counts; it is not a leave-one-out GWAS or a test of model stability after refitting.

In [ ]:
lines = ["| Region | Full case-minus-control ALT frequency | Range after omitting one case | Carrier-pair mean similarity | All-pair mean similarity |",
         "| --- | ---: | ---: | ---: | ---: |"]
def value_or_blank(value):
    return "Unavailable" if value is None else f"{value:.3f}"
for region in review["regions"]:
    lines.append(f"| {region['region']} | {region['case_control_ALT_frequency_difference']:.3f} | "
                 f"{region['leave_one_case_out_frequency_difference_min']:.3f} to {region['leave_one_case_out_frequency_difference_max']:.3f} | "
                 f"{value_or_blank(region['carrier_pair_relationship']['mean'])} | {value_or_blank(region['all_pair_relationship']['mean'])} |")
display(Markdown("\n".join(lines)))

## What this review establishes

This chapter provides regional measured-marker support, observed genotype and carrier summaries, and explicit fit comparisons for manual assessment. It does not establish that a signal is independent of all family structure, validate genome-wide tail probabilities, or identify a causal variant. Small case groups and long correlated regions remain limitations.

Biological annotation and reference-assembly conversion for interpretation follow separately. Precise fine-mapping or causal credible sets are not attempted at this stage. Supporting details are in the [candidate-region methods note](../docs/methods/09%20Candidate%20region%20review.md).